# FhalconFood · Baseline integrado (Semana 4)
**Proyecto de titulación MIA — Grupo 5 | Peter Cañizares y Henry Mantilla**

Prototipo inicial reproducible para **Google Colab**. Implementa la alternativa A: reglas deterministas, búsqueda controlada en catálogo, plantillas y revisión humana. **No utiliza BETO, LLM ni K-means**, que corresponden a etapas posteriores cuando existan datos reales suficientes.

**Fuente:** `LISTADO PRODUCTOS SEP26.xlsx`, hoja `LISTA DE PRECIOS`. El nombre de la hoja **no demuestra** que existan precios, stock o condiciones vigentes; el notebook no los inventa. No se han entrenado modelos ni medido resultados comerciales.

**Instrucciones:** ejecutar las celdas en orden; cuando se solicite, subir el Excel original. También funciona fuera de Colab si se coloca el archivo en el directorio actual.

## 1. Dependencias, parámetros y trazabilidad

In [ ]:
import os, re, json, hashlib, unicodedata, sys, platform
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

SEED = 42
ARCHIVO = 'LISTADO PRODUCTOS SEP26.xlsx'
HOJA = 'LISTA DE PRECIOS'
VERSION_REGLAS = 'baseline-s4-v1.0'
FECHA_EJECUCION = datetime.now(timezone.utc).isoformat()
print('Python:', platform.python_version(), '| pandas:', pd.__version__, '| reglas:', VERSION_REGLAS)


## 2. Catálogo Excel incluido y carga automática
El archivo **LISTADO PRODUCTOS SEP26.xlsx** está integrado en el notebook. Ejecuta las celdas en orden; **no es necesario subirlo por separado**. Se comprueba su integridad mediante SHA-256.


In [ ]:
# Catálogo Excel incorporado en este notebook: no hace falta subir archivos.
import base64, hashlib
from pathlib import Path
ARCHIVO = 'LISTADO PRODUCTOS SEP26.xlsx'
DATOS_EXCEL_B64 = (
    "..."
)
contenido = base64.b64decode(DATOS_EXCEL_B64)
assert hashlib.sha256(contenido).hexdigest() == '3f887f54409dbb3cb267355a4713707468877bd568924394a2af8de8dbdcf5ac', 'El catálogo integrado no superó la verificación de integridad'
Path(ARCHIVO).write_bytes(contenido)
print(f'Catálogo Excel integrado: {ARCHIVO} ({len(contenido):,} bytes)')
del DATOS_EXCEL_B64, contenido


In [ ]:
if not Path(ARCHIVO).exists():
    try:
        from google.colab import files
        print('Sube el archivo original:', ARCHIVO)
        subidos = files.upload()
        if ARCHIVO not in subidos:
            raise FileNotFoundError('Se requiere el Excel original con nombre: '+ARCHIVO)
    except ImportError:
        raise FileNotFoundError('Coloca el archivo Excel junto al notebook y vuelve a ejecutar.')

with open(ARCHIVO,'rb') as f:
    hash_archivo = hashlib.sha256(f.read()).hexdigest()
crudo = pd.read_excel(ARCHIVO, sheet_name=HOJA, header=None, dtype=str)
print('Dimensiones originales:', crudo.shape, '| SHA-256:', hash_archivo)
display(crudo.iloc[:6,:min(6,crudo.shape[1])])


## 3. Preparación: de categorías por columna a catálogo normalizado
La fila 2 de la hoja (índice 1) contiene encabezados de categorías; los productos comienzan en la fila 3. Se omiten celdas vacías, sin interpretar cantidades o precios inexistentes.

In [ ]:
def limpiar_espacios(s):
    return re.sub(r'\s+', ' ', str(s)).strip()

def clave(s):
    s = unicodedata.normalize('NFKD', str(s))
    s = ''.join(c for c in s if not unicodedata.combining(c))
    return limpiar_espacios(s).casefold()

def normalizar_catalogo(df):
    registros = []
    for j in range(df.shape[1]):
        cat = df.iat[1,j] if df.shape[0]>1 else None
        if pd.isna(cat) or not limpiar_espacios(cat):
            continue
        categoria = limpiar_espacios(cat)
        for i in range(2,df.shape[0]):
            valor = df.iat[i,j]
            if pd.isna(valor) or not limpiar_espacios(valor):
                continue
            producto = limpiar_espacios(valor)
            registros.append({'categoria':categoria,'producto':producto,
                              'fila_excel':i+1,'columna_excel':j+1})
    tabla = pd.DataFrame(registros)
    if tabla.empty:
        raise ValueError('No se identificaron productos: revisar la estructura del Excel.')
    tabla['categoria_clave'] = tabla['categoria'].map(clave)
    tabla['producto_clave'] = tabla['producto'].map(clave)
    antes = len(tabla)
    tabla = tabla.drop_duplicates(['categoria_clave','producto_clave']).reset_index(drop=True)
    tabla['id_producto'] = tabla.apply(lambda r: hashlib.sha256((r['categoria_clave']+'|'+r['producto_clave']).encode()).hexdigest()[:12],axis=1)
    return tabla, antes-len(tabla)

catalogo, duplicados = normalizar_catalogo(crudo)
print('Productos únicos:',len(catalogo),'| Categorías:',catalogo.categoria_clave.nunique(),'| Duplicados retirados:',duplicados)
display(catalogo[['id_producto','categoria','producto']].head(12))


## 4. Motor de búsqueda y controles de información
La coincidencia es determinista y no utiliza modelos externos. Si no hay evidencia suficiente, no confirma información. Las búsquedas pueden devolver coincidencias aproximadas; siempre requieren revisión humana antes de publicarse.

In [ ]:
PALABRAS_VACIAS = {'de','la','el','los','las','un','una','y','para','por','con','que','del','en','quiero','necesito','tienen','tiene','hay','busco','precio','cuanto','cuesta','disponible','stock','comprar','pedido','especial','condicion','condiciones','comercial','comerciales'}
def tokens(texto):
    return {t for t in re.findall(r'[a-z0-9]+',clave(texto)) if len(t)>2 and t not in PALABRAS_VACIAS}

def buscar_productos(consulta, limite=5):
    q = tokens(consulta)
    if not q:
        return catalogo.iloc[0:0][['id_producto','categoria','producto']].copy()
    def puntuar(r):
        p = tokens(r['producto']); c = tokens(r['categoria'])
        return 3*len(q & p)+len(q & c)
    t = catalogo.copy()
    t['puntuacion'] = t.apply(puntuar,axis=1)
    t = t[t.puntuacion>0].sort_values(['puntuacion','producto'],ascending=[False,True])
    return t[['id_producto','categoria','producto','puntuacion']].head(limite)

consulta_ejemplo = catalogo.iloc[0]['producto']
print('Consulta de ejemplo tomada del Excel:',consulta_ejemplo)
display(buscar_productos(consulta_ejemplo))


## 5. Módulo A — campañas digitales con plantillas
Produce un **borrador** a partir de un producto real, sin prometer descuentos, disponibilidad ni precios.

In [ ]:
def campana_producto(id_producto, canal='WhatsApp'):
    fila = catalogo[catalogo.id_producto==id_producto]
    if fila.empty:
        return {'estado':'requiere_revision','mensaje':'ID de producto no encontrado.'}
    r = fila.iloc[0]
    if canal not in ('WhatsApp','Correo','Redes sociales'):
        return {'estado':'requiere_revision','mensaje':'Canal no autorizado en la demostración.'}
    texto = (f'FhalconFood | {r["producto"]}. Categoría: {r["categoria"]}. '
             'Solicita a nuestro equipo comercial información actualizada y condiciones de compra. '
             'La disponibilidad y el precio deben confirmarse con un asesor.')
    return {'estado':'borrador_pendiente_revision','canal':canal,'id_producto':id_producto,'contenido':texto}

display(campana_producto(catalogo.iloc[0].id_producto))


## 6. Módulo B — agrupación descriptiva por categorías
**No es segmentación de clientes**: el Excel no contiene clientes ni compras. La agrupación por categorías permite demostrar el módulo sin afirmar resultados de K-means.

In [ ]:
def grupos_catalogo(limite=15):
    return (catalogo.groupby('categoria',as_index=False)
            .agg(numero_productos=('id_producto','count'))
            .sort_values(['numero_productos','categoria'],ascending=[False,True])
            .head(limite))
display(grupos_catalogo())


## 7. Módulo C — recomendación basada en coincidencia de términos
Es un ranking determinista de productos del catálogo, **no** una recomendación personalizada ni una predicción de compra.

In [ ]:
def recomendar(consulta, limite=5):
    hallados = buscar_productos(consulta,limite)
    return {'estado':'coincidencias_catalogo' if len(hallados) else 'sin_coincidencias_derivar',
            'consulta':consulta,'resultados':hallados.to_dict('records'),
            'nota':'Validar pertinencia y condiciones comerciales con el equipo humano.'}
print(json.dumps(recomendar(consulta_ejemplo),ensure_ascii=False,indent=2))


## 8. Módulo D — chatbot con clasificación por reglas y derivación humana
La intención se infiere con palabras clave; las solicitudes de precio, inventario y compra se derivan. No hay conversaciones reales ni entrenamiento supervisado.

In [ ]:
REGLAS_INTENCION = {
 'compra': ('comprar','pedido','cotizacion','cotizar','orden de compra','facturar','condicion especial','negociar','descuento'),
 'precio': ('precio','cuanto cuesta','valor','costo','tarifa'),
 'disponibilidad': ('stock','disponible','existencia','inventario','hay unidades'),
 'catalogo': ('catalogo','producto','tienen','ofrecen','busco','necesito')
}

def detectar_intencion(mensaje):
    s = clave(mensaje)
    for etiqueta in ('compra','precio','disponibilidad','catalogo'):
        if any(clave(p) in s for p in REGLAS_INTENCION[etiqueta]):
            return etiqueta
    return 'desconocida'

def chatbot(mensaje):
    intencion = detectar_intencion(mensaje)
    encontrados = buscar_productos(mensaje,3)
    items = encontrados[['id_producto','producto','categoria']].to_dict('records')
    if intencion == 'compra':
        respuesta = 'Puedo ayudarte a contactar a un asesor para continuar con tu solicitud de compra.'
    elif intencion == 'precio':
        respuesta = 'No tengo precios vigentes verificados. Un asesor debe confirmar el precio y las condiciones comerciales.'
    elif intencion == 'disponibilidad':
        respuesta = 'No tengo inventario vigente verificado. Un asesor debe confirmar la disponibilidad.'
    elif intencion == 'catalogo' and items:
        respuesta = 'Encontré posibles coincidencias en el catálogo. Un asesor puede confirmar detalles y condiciones.'
    else:
        respuesta = 'No puedo confirmar la respuesta con la información disponible. Te derivaré a un asesor.'
    return {'intencion_regla':intencion,'respuesta':respuesta,'coincidencias':items,
            'derivar_humano':intencion!='catalogo' or not bool(items),
            'venta_automatica':False}

for ejemplo in ['¿Cuál es el precio?', '¿Hay stock?', 'Quiero hacer un pedido',f'Busco {consulta_ejemplo}', 'Necesito una condición especial']:
    print('Cliente:',ejemplo,'\n',json.dumps(chatbot(ejemplo),ensure_ascii=False,indent=2),'\n')


## 9. Pruebas funcionales reproducibles (no métricas predictivas)
Estas pruebas comprueban reglas de seguridad y trazabilidad. **No son accuracy, F1 ni métricas de negocio**.

In [ ]:
pruebas = []
def verificar(nombre, condicion):
    pruebas.append({'prueba':nombre,'resultado':'OK' if condicion else 'FALLA'})

verificar('IDs de producto únicos',catalogo.id_producto.is_unique)
verificar('Catálogo no vacío',len(catalogo)>0)
verificar('No inventa precio', 'no tengo precios' in clave(chatbot('¿Cuál es el precio?')['respuesta']))
verificar('Deriva compra',chatbot('Quiero comprar')['derivar_humano'])
verificar('Deriva disponibilidad',chatbot('¿Hay stock?')['derivar_humano'])
verificar('No ejecuta ventas',not chatbot('Quiero comprar')['venta_automatica'])
verificar('Campaña requiere revisión',campana_producto(catalogo.iloc[0].id_producto)['estado']=='borrador_pendiente_revision')
verificar('Recomendación devuelve datos reales',all(x['id_producto'] in set(catalogo.id_producto) for x in recomendar(consulta_ejemplo)['resultados']))
resultado_pruebas = pd.DataFrame(pruebas)
display(resultado_pruebas)
assert (resultado_pruebas.resultado=='OK').all(), 'Hay pruebas funcionales fallidas.'
print('Pruebas funcionales superadas:',len(pruebas))


## 10. Registro, exportación y próximos experimentos
Los resultados exportados son **salidas del prototipo**, no resultados de evaluación con clientes reales. La fase futura requerirá mensajes etiquetados, históricos de campañas, clientes autorizados, particiones y métricas por módulo.

In [ ]:
salida = Path('salidas_baseline_s4')
salida.mkdir(exist_ok=True)
catalogo[['id_producto','categoria','producto','fila_excel','columna_excel']].to_csv(salida/'catalogo_normalizado.csv',index=False,encoding='utf-8-sig')
resultado_pruebas.to_csv(salida/'pruebas_funcionales.csv',index=False,encoding='utf-8-sig')
registro = {
 'proyecto':'FhalconFood baseline Semana 4','version_reglas':VERSION_REGLAS,
 'fecha_utc':FECHA_EJECUCION,'archivo_origen':ARCHIVO,'hoja':HOJA,
 'sha256':hash_archivo,'filas_crudas':int(crudo.shape[0]),'columnas_crudas':int(crudo.shape[1]),
 'productos_unicos':int(len(catalogo)),'categorias':int(catalogo.categoria_clave.nunique()),
 'duplicados_eliminados':int(duplicados),'semilla_futura':SEED,
 'python':platform.python_version(),'pandas':pd.__version__,
 'entrenamiento_realizado':False,'metricas_predictivas':None,
 'limitaciones':['Sin mensajes reales','Sin historial de campañas','Sin perfiles de clientes',
                 'Sin precios ni stock validados','Revisión humana obligatoria']}
with open(salida/'registro_ejecucion.json','w',encoding='utf-8') as f:
    json.dump(registro,f,ensure_ascii=False,indent=2)
print('Archivos generados:',*[str(p) for p in salida.iterdir()],sep='\n - ')


### Plan de evolución (no ejecutado)
1. **Campañas:** comparar plantillas con generación asistida por LLM, con revisión humana ciega, pertinencia, tiempo de elaboración y errores críticos.
2. **Segmentación:** cuando existan perfiles de clientes, comparar reglas de negocio frente a K-means y validar interpretabilidad con el equipo comercial.
3. **Recomendación:** cuando existan interacciones y campañas históricas, comparar ranking por reglas frente a modelos supervisados con métricas como Precision@k.
4. **Chatbot:** obtener mensajes reales autorizados y etiquetados; comparar reglas con TF-IDF + regresión logística (baseline estadístico adicional) y BETO. Usar particiones agrupadas por conversación/cliente y F1 macro, recall de intención de compra y tasa de derivación correcta.

**Seguridad y reproducibilidad:** versionar catálogo, reglas, librerías y resultados; no publicar respuestas sin aprobación comercial; no usar datos personales sin autorización; no ajustar modelos sobre el conjunto de prueba.